# Patron de Reflexion: un agente que se autocritica

Este notebook muestra el patron de **reflexion** (generar -> criticar -> revisar): un LLM produce una respuesta, otro rol del mismo modelo (el "revisor") la critica, y si encuentra un problema el modelo la corrige. Se repite hasta que el revisor la aprueba o se agotan los intentos.

Caso de uso: un agente que explica conceptos de fisica a un estudiante, y se autocorrige antes de entregar la respuesta final.

Las dos "voces" (explicador y revisor) son el mismo LLM con distintas instrucciones de sistema.

Requisitos: Ollama corriendo localmente con al menos el modelo `llama3.2:1b` descargado (ver `ayuda.txt`).

In [1]:
from reflection_agent import explicar, revisar, corregir, run_reflection
from display_helpers import show_trace, dMarkdown

## 1. Sin reflexion (linea base)

Primero pedimos la explicacion una sola vez, sin ningun paso de revision. Con un modelo pequeno como `llama3.2:1b`, a veces la explicacion tiene errores o imprecisiones que pasan sin que nadie las revise.

In [3]:
model = "llama3.2:1b"
concepto = "No es posible llegar a cero absoluto, pero ¿podemos llegar a una temperatura absoluta negativa en un sistema cuantico? Pista: Que ocurre con la beta termodinámica o temperatura inversa."

explicacion_base = explicar(concepto, model)
dMarkdown(explicacion_base)

Claro, aquí te explico cómo funciona. La idea de llegar a una temperatura absoluta negativa en un sistema cuantico es un poco complicada, pero vamos a ver cómo se aborda en la física. Es importante destacar que la física clásica no tiene restricciones para la temperatura, pero la física cuántica la tiene.

La física cuántica es un campo de estudio que se enfoca en los fenómenos que ocurren en el nivel de las partículas más pequeñas, como átomos y electrones. En este nivel, la temperatura no es una cantidad física estándar, sino más bien una medida de la energía dispersa por las partículas.

Una de las propiedades fundamentales de la física cuántica es la conservación de la energía. En otras palabras, la energía no puede ser creada ni destruida, solo se transfiere de una parte a otra. Esto significa que, en general, no hay una temperatura absoluta negativa, ya que la energía no puede disiparse en una dirección.

Sin embargo, hay un concepto llamado "temperatura negativa" en el contexto de la física cuántica. Esta temperatura se relaciona con la temperatura inversa, que es la temperatura a la que el sistema puede ser más caluroso que la temperatura absoluta cero. Esto se logra cuando los electrones y las partículas en el sistema se mueven de manera que puedan transferir mucha energía de manera eficiente.

En este contexto, la "temperatura negativa" se refiere a la temperatura a la que el sistema puede ser más caluroso que la temperatura absoluta negativa zero, que no tiene sentido en la física clásica. Sin embargo, esta temperatura negativa es una propuesta teórica que se ha discutido en la física cuántica y se refiere a la temperatura a la que el sistema puede ser más caluroso que la temperatura absoluta negativa zero.

En resumen, no es posible llegar a una temperatura absoluta negativa en un sistema cuantico, pero sí se pueden lograr temperaturas negativas en el contexto de la física cuántica, especialmente en sistemas que pueden transferir mucha energía de manera eficiente.

## 2. El ciclo paso a paso

El patron tiene tres funciones en `reflection_agent.py`:

- `explicar(concepto, model)` -- genera una primera explicacion.
- `revisar(concepto, explicacion, model)` -- el mismo modelo, con otra instruccion de sistema, actua como revisor: dice si la aprueba y por que no si la rechaza.
- `corregir(concepto, explicacion, comentarios, model)` -- vuelve a generar la explicacion, esta vez viendo los comentarios del revisor.

Vamos a correrlas a mano una vez, sobre la explicacion de la seccion anterior.

In [4]:
revision = revisar(concepto, explicacion_base, model)
print("Aprobada:", revision["aprobada"])
dMarkdown(revision["comentarios"])

Aprobada: False


APROBADA

RECHAZADA

Que la física cuántica tenga restricciones para la temperatura es un planteamiento impreciso, ya que la física cuántica no tiene restricciones para la temperatura en el nivel de las partículas más pequeñas. En realidad, la conservación de la energía es una propiedad fundamental de la física cuántica, y no hay una temperatura absoluta negativa en el sentido tradicional.

La "temperatura negativa" se refiere a la temperatura inversa, que es la temperatura a la que el sistema puede ser más caluroso que la temperatura absoluta negativa zero, que no tiene sentido en la física clásica. Sin embargo, esta temperatura negativa es una propuesta teórica que se ha discutido en la física cuántica y no está respaldada por la física clásica.

En resumen, la física cuántica no tiene restricciones para la temperatura en el nivel de las partículas más pequeñas, y no es posible llegar a una temperatura absoluta negativa en un sistema cuantico, pero sí se pueden lograr temperaturas negativas en el contexto de la física cuántica.

In [5]:
if not revision["aprobada"]:
    explicacion_corregida = corregir(concepto, explicacion_base, revision["comentarios"], model)
    dMarkdown(explicacion_corregida)
else:
    dMarkdown("El revisor la aprobo, no hace falta corregir.")

Me disculpo por el error. Aquí te dejo la respuesta corregida:

APROBADA

RECHAZADA

La física cuántica no tiene restricciones fundamentales para la temperatura en el nivel de las partículas más pequeñas, como átomos y electrones. La conservación de la energía es una propiedad básica de la física cuántica, y no hay una temperatura absoluta negativa en el sentido tradicional.

La "temperatura negativa" se refiere a la temperatura inversa, que es la temperatura a la que el sistema puede ser más caluroso que la temperatura absoluta negativa cero, que es un concepto hipotético y no respaldado por la física clásica. Sin embargo, esta temperatura negativa es una propuesta teórica que se ha discutido en la física cuántica y no está respaldada por la física clásica.

En resumen, la física cuántica no tiene restricciones para la temperatura en el nivel de las partículas más pequeñas, y no es posible llegar a una temperatura absoluta negativa en un sistema cuantico. Sin embargo, se pueden lograr temperaturas negativas en el contexto de la física cuántica, especialmente en sistemas que pueden transferir mucha energía de manera eficiente.

La física cuántica es un campo de estudio que se enfoca en los fenómenos que ocurren en el nivel de las partículas más pequeñas, y no está limitada por la conservación de la energía en este nivel. La temperatura es una medida de la energía dispersada por las partículas, y no es una propiedad fundamental de la física cuántica en el sentido tradicional.

## 3. El ciclo completo: `run_reflection`

`run_reflection` encadena estos tres pasos automaticamente hasta `max_rounds` veces, o hasta que el revisor aprueba. Devuelve la traza completa (cada ronda con su explicacion y su revision) para poder inspeccionar que paso.

In [6]:
trace = run_reflection(concepto, model=model, max_rounds=3)
show_trace(concepto, trace)

## Concepto: No es posible llegar a cero absoluto, pero ¿podemos llegar a una temperatura absoluta negativa en un sistema cuantico? Pista: Que ocurre con la beta termodinámica o temperatura inversa.
### Ronda 1
En física, la idea de llegar a cero absoluto es un poco extraña, ya que la temperatura se refiere a la energía que se libera en un sistema cuando se calienta o fríe. En el caso de un sistema cuantico, la energía no se puede "calentar" o "fríe" de la misma manera que un cuerpo físico. Sin embargo, hay algo interesante que puede pasar en un sistema cuantico.

En la beta termodinámica, la temperatura es medida como la energía distribuida entre todas las partículas en el sistema. Si nos suministramos energía a un sistema cuantico en modo no estacionario, podemos aumentar su temperatura. Sin embargo, esta forma de aumentar la temperatura no implica que el sistema alcance una temperatura absoluta negativa, sino que simplemente aumenta su energía distribuida en el espacio.

La temperatura inversa, que se puede medir en un sistema cuantico, es una medida de la energía distribuida en el espacio. En otras palabras, si nos suministramos energía a un sistema cuantico en modo no estacionario, podemos aumentar su energía distribuida en el espacio. Sin embargo, esta forma de aumentar la energía distribuida no implica que el sistema alcance una temperatura absoluta negativa, sino que simplemente aumenta su energía distribuida en el espacio.

En resumen, en un sistema cuantico, podemos aumentar la energía distribuida en el espacio, pero no podemos alcanzar una temperatura absoluta negativa.

**Revisión del crítico -- Rechazada, se corrige en la siguiente ronda**<br>
RECHAZADA

La respuesta anterior no proporciona una explicación clara y concisa sobre la relación entre la temperatura absoluta y la energía distribuida en un sistema cuantico. La concepto de "temperatura absoluta negativa" es fundamental en la física cuántica y se refiere a la posibilidad de alcanzar una temperatura negativa en un sistema cuantico.

En la física cuántica, la temperatura absoluta se refiere a la energía distribuida en el espacio de un sistema, que es una medida estadística de la disposición de las partículas en el sistema. Una temperatura absoluta negativa sería una medida de la energía total del sistema que fuera negativa.

Sin embargo, la respuesta del profesor destaca que el sistema cuantico no puede alcanzar una temperatura absoluta negativa debido a la forma en que se suministra energía a los sistemas no estacionarios. En otras palabras, la forma en que se mide la temperatura inversa, en lugar de ser una medida de energía absoluta, es una medida de energía distribuida en el espacio.

No se proporciona una explicación clara y concisa sobre la relación entre la temperatura absoluta y la energía distribuida en un sistema cuantico, lo que lleva a la pregunta de si se puede alcanzar una temperatura absoluta negativa. Se recomienda revisar la explicación del profesor para asegurarse de que sea precisa y clara.

---
### Ronda 2
Entiendo la crítica y voy a corregir mi explicación.

En física, la temperatura absoluta se refiere a la energía distribuida en el espacio de un sistema, que es una medida estadística de la disposición de las partículas en el sistema. En otras palabras, es la medida de la energía total del sistema que se distribuye uniformemente en el espacio.

Una temperatura absoluta negativa sería una medida de la energía total del sistema que fuera negativa, lo que no está relacionado con la forma en que se suministra energía a los sistemas no estacionarios. En realidad, la forma en que se mide la temperatura inversa, en lugar de ser una medida de energía absoluta, es una medida de la energía distribuida en el espacio.

En un sistema cuantico, cuando se suministra energía a un sistema en modo no estacionario, la temperatura inversa se considera una medida de la energía distribuida en el espacio. Sin embargo, no significa que el sistema alcance una temperatura negativa absoluta. En su lugar, la energía distribuida en el espacio se refiere a la energía total del sistema, que no es negativa.

En otras palabras, la temperatura inversa es una medida de la energía distribuida en el espacio, pero no es una medida de la temperatura absoluta negativa. La temperatura absoluta negativa no es una propiedad que se pueda alcanzar en un sistema cuantico debido a la forma en que se suministra energía a los sistemas no estacionarios.

Espero que esta versión corregida sea más clara y precisa.

**Revisión del crítico -- Aprobada**<br>
APROBADA

La crítica de la explicación es la siguiente:

* La frase "En otras palabras, la temperatura inversa es una medida de la energía distribuida en el espacio, pero no es una medida de la temperatura absoluta negativa" podría ser más concisa y directa.
* La oración "Sin embargo, no significa que el sistema alcance una temperatura negativa absoluta" podría estar un poco vaga, dado que la temperatura absoluta no tiene sentido en un sistema cuantico.
* La frase "La temperatura absoluta negativa no es una propiedad que se pueda alcanzar en un sistema cuantico debido a la forma en que se suministra energía a los sistemas no estacionarios" podría ser un poco ambigua, ya que la forma en que se suministra energía a los sistemas no estacionarios no está relacionada con la temperatura absoluta negativa.

Ninguno

---

## 4. Varios conceptos

Probemos con un par de conceptos mas; el codigo no cambia, solo el texto que le pasamos.

In [8]:
conceptos = [
    "¿Existe la temperatura absoluta negativa?",
    "¿Existe el cero absoluto?"
]

for c in conceptos:
    trace = run_reflection(c, model=model, max_rounds=3)
    show_trace(c, trace)

## Concepto: ¿Existe la temperatura absoluta negativa?
### Ronda 1
La temperatura absoluta es un concepto fundamental en la física, y la respuesta es un poco complicada. En el sentido más amplio, la temperatura absoluta es la temperatura en cero Kelvin (K), que es la temperatura absoluta en la unidad de cero absoluto. La cero absoluta es un concepto muy abstracto, y no tiene un valor físico exacto.

Sin embargo, en el sentido más específico, la temperatura absoluta no tiene una "negativa" en el sentido tradicional. La temperatura no puede ser menor o mayor que cero, ya que eso sería una contradicción. La temperatura es una medida de la energía distribuida en un sistema, y no tiene un valor absoluto. Pero, en teoría, si tomamos la temperatura de un sistema en cero absoluto y la convierte en una temperatura absoluta, podríamos decir que la temperatura absoluta es negativa. Esto se debe a que la temperatura absoluta es una medida de la energía distribuida en el sistema, y si la energía es cero, entonces la temperatura es cero.

Sin embargo, es importante destacar que este concepto no es ampliamente aceptado en la física moderna. En la mayoría de los sistemas físicos, la temperatura no es una variable negativa, y no se puede tomar una temperatura en cero absoluto y la convierte en una temperatura negativa.

**Revisión del crítico -- Aprobada**<br>
APROBADA

La respuesta anterior presenta algunos errores conceptuales:

* La afirmación "La cero absoluta es un concepto muy abstracto, y no tiene un valor físico exacto" es correcta, pero no se puede decir que la cero absoluta no tiene un valor físico exacto, sino que su valor exacto es cero.
* La afirmación "La temperatura absoluta no tiene una 'negativa' en el sentido tradicional" es correcta, pero no se puede decir que la temperatura absoluta no tiene una "negativa" en el sentido de que la energía no puede ser menor o mayor que cero.
* La afirmación "La temperatura absoluta es una medida de la energía distribuida en el sistema" es correcta, pero no se puede decir que esta es la razón por la que la temperatura absoluta no puede ser negativa.

Sin embargo, la respuesta anterior sigue siendo un poco ambigua y no presenta una explicación clara y concisa del concepto de temperatura absoluta. Además, la afirmación "En teoría, si tomamos la temperatura de un sistema en cero absoluto y la convierte en una temperatura absoluta, podríamos decir que la temperatura absoluta es negativa" es un concepto complejo que requiere una explicación más detallada y técnica.

Ninguno

---

## Concepto: ¿Existe el cero absoluto?
### Ronda 1
Excelente pregunta. El concepto de cero absoluto es un tema complejo y controvertido en la física. En general, se considera que el cero absoluto es la ubicación en un espacio de medida, que es la distancia desde el origen (0,0,0) en un sistema de coordenadas.

Sin embargo, en la teoría de los números, el cero absoluto no existe como un valor numérico. Los números son conceptos que se definen en términos de relaciones entre otras unidades, y el cero no es una unidad de medida. Pero, en la teoría de la relatividad, se introduce la noción de un cero absoluto, que se refiere a una ubicación en el espacio, como mencioné anteriormente.

En cuanto a la identidad del cero absoluto, algunos físicos consideran que existe como una ubicación en el espacio, como la ubicación del origen (0,0,0). Otros, por otro lado, argumentan que no existe como un valor numérico, ya que no puede ser representado por una unidad de medida.

En resumen, la existencia del cero absoluto es un tema discutido y controvertido en la física, y no hay un consenso claro sobre su naturaleza.

**Revisión del crítico -- Rechazada, se corrige en la siguiente ronda**<br>
RECHAZADA

La expresión "cero absoluto" se usa comúnmente para referirse a la ubicación en un espacio, lo que sugiere que existe en un espacio de medida. Sin embargo, el profesor menciona que en la teoría de los números, el cero absoluto no existe como un valor numérico, lo que implica que la ubicación del cero absoluto no puede ser representada como una unidad de medida. Esta afirmación parece contradictoria con la afirmación del profesor sobre la existencia de una ubicación del origen (0,0,0) como cero absoluto en la teoría de la relatividad.

---
### Ronda 2
Me disculpo por el error. La explicación original contenía una afirmación contradictoria. En efecto, en la teoría de los números, el cero absoluto no existe como un valor numérico, y en la teoría de la relatividad, se refiere a una ubicación en el espacio, como la ubicación del origen (0,0,0).

En este contexto, la ubicación del origen (0,0,0) es efectivamente una ubicación que se puede considerar como el cero absoluto en la teoría de la relatividad. Por otro lado, la ubicación del cero absoluto en el espacio de medida se refiere a la distancia desde el origen (0,0,0) en un sistema de coordenadas.

En resumen, la ubicación del origen (0,0,0) es efectivamente el cero absoluto en la teoría de la relatividad, pero no la ubicación del cero absoluto en el espacio de medida. La ubicación del cero absoluto en el espacio de medida no existe como un valor numérico, ya que no puede ser representado por una unidad de medida.

Me aseguraré de ser más preciso en el futuro y evitar afirmaciones contradictorias. Gracias por señalar el error.

**Revisión del crítico -- Rechazada, se corrige en la siguiente ronda**<br>
APROBADA

RECHAZADA

Ninguno.

---
### Ronda 3
Me disculpo por el error. La explicación original contenía una afirmación contradictoria. En efecto, en la teoría de los números, el cero absoluto no existe como un valor numérico, y en la teoría de la relatividad, se refiere a una ubicación en el espacio, como la ubicación del origen (0,0,0).

En este contexto, la ubicación del origen (0,0,0) es efectivamente el cero absoluto en la teoría de la relatividad. Por otro lado, la ubicación del cero absoluto en el espacio de medida se refiere a la distancia desde el origen (0,0,0) en un sistema de coordenadas.

En resumen, la ubicación del origen (0,0,0) es efectivamente el cero absoluto en la teoría de la relatividad, pero la ubicación del cero absoluto en el espacio de medida no existe como un valor numérico, ya que no puede ser representado por una unidad de medida.

Me aseguraré de ser más preciso en el futuro y evitar afirmaciones contradictorias. Gracias por señalar el error.

**Revisión del crítico -- Aprobada**<br>
APROBADA

La afirmación original contenía una afirmación contradictoria, ya que el cero absoluto en la teoría de la relatividad se refiere a una ubicación en el espacio, mientras que la ubicación del cero absoluto en el espacio de medida se refiere a la distancia desde el origen (0,0,0) en un sistema de coordenadas.

---

## 5. Modelo pequeño vs modelo más grande

`llama3.2:1b` es rápido, pero poco confiable, y eso se nota **en los dos roles**: el explicador puede cometer errores, y el revisor puede no detectarlos (o rechazar explicaciones que sí estaban bien). Un modelo más grande, como `qwen3:1.7b`, suele ser más consistente en ambos roles -- sin tocar el resto del código, solo cambia el parámetro `model`.

In [9]:
model_grande = "qwen3:1.7b"
trace = run_reflection(concepto, model=model_grande, max_rounds=3)
show_trace(concepto, trace)

## Concepto: No es posible llegar a cero absoluto, pero ¿podemos llegar a una temperatura absoluta negativa en un sistema cuantico? Pista: Que ocurre con la beta termodinámica o temperatura inversa.
### Ronda 1
En sistemas cuánticos, la temperatura puede ser negativa debido a que la distribución de energía en los niveles de energía está caracterizada por un término que puede ser negativo. Esto se debe a que, en algunos sistemas, la energía promedio de los partículas está dominada por niveles de energía altos, lo que hace que la temperatura (como la inversa de la beta termodinámica) sea negativa. Aunque la temperatura absoluta (0 K) es imposible en sistemas clásicos, en sistemas cuánticos se puede alcanzar una temperatura negativa, aunque no sea absoluta. Esta situación refleja la naturaleza de la termodinámica cuántica, donde la temperatura puede tener valores negativos en ciertos estados de equilibrio.

**Revisión del crítico -- Rechazada, se corrige en la siguiente ronda**<br>
RECHAZADA

COMENTARIOS: La explicación confunde el concepto de temperatura absoluta (0 K) con el de temperatura negativa en sistemas cuánticos. Aunque en sistemas cuánticos se pueden alcanzar temperaturas negativas (no absolutas), la "temperatura absoluta negativa" es un concepto inválido en termodinámica clásica. Además, la frase "no es posible llegar a una temperatura absoluta negativa" es incorrecta, ya que los sistemas cuánticos no alcanzan la temperatura absoluta (0 K), pero sí pueden tener temperaturas negativas (no absolutas).

---
### Ronda 2
La temperatura absoluta (0 K) es un límite teórico que no se puede alcanzar en ningún sistema físico, ya sea clásico o cuántico. En sistemas cuánticos, aunque la temperatura (como la inversa de la beta termodinámica) puede ser negativa (no absoluta), este concepto no se puede referir a una temperatura absoluta negativa. La temperatura negativa en sistemas cuánticos es un fenómeno que no se corresponde con la termodinámica clásica, donde la temperatura siempre es positiva. Por lo tanto, aunque en sistemas cuánticos se pueden obtener valores negativos de la temperatura termodinámica, estos no son equivalentes a una temperatura absoluta negativa, y la idea de "no poder llegar a una temperatura absoluta negativa" es incorrecta, ya que los sistemas cuánticos no alcanzan la temperatura absoluta (0 K), pero sí pueden tener temperaturas negativas en ciertos estados.

**Revisión del crítico -- Rechazada, se corrige en la siguiente ronda**<br>
<RECHAZADA>  
El profesor confunde el concepto de temperatura negativa en sistemas cuánticos con la temperatura absoluta negativa, aunque esta última no es posible. La explicación no clarifica que en sistemas cuánticos se pueden alcanzar temperaturas negativas (como la inversa de la beta termodinámica), pero estos no son equivalentes a una temperatura absoluta negativa. Además, la frase "no poder llegar a cero absoluto" es correcta, pero la respuesta no aborda la distinción entre temperaturas negativas y cero absoluto.

---
### Ronda 3
La temperatura absoluta (0 K) es un límite teórico que no se puede alcanzar en ningún sistema físico, incluyendo los cuánticos. En sistemas cuánticos, aunque la temperatura termodinámica (como la inversa de la beta termodinámica) puede ser negativa, estos valores no corresponden a una temperatura absoluta negativa. La temperatura negativa en sistemas cuánticos se refiere a un estado de equilibrio donde el sistema tiende a absorber energía, lo cual no se alinea con la termodinámica clásica. Aunque en sistemas cuánticos se pueden obtener valores negativos de la temperatura termodinámica, estos no son equivalentes a la temperatura absoluta negativa, ya que la temperatura absoluta es un escalado del entropía en la termodinámica clásica. La afirmación de que "no se puede llegar a cero absoluto" es correcta, pero la respuesta debe enfatizar la distinción entre temperaturas negativas (en el sentido termodínámico) y la temperatura absoluta negativa, que no es física.

**Revisión del crítico -- Aprobada**<br>
<APROBADA>  
COMENTARIOS: La explicación es precisa y clara, destacando la distinción entre temperaturas negativas en el sentido termodínámico y la temperatura absoluta negativa. La referencia a la beta termodinámica y el uso de términos técnicos son adecuados y no generan confusión.

---